# Text Preprocessing for 10-K Sections

Produces **two text variants** from the raw extracted sections (item\_1, item\_1A, item\_7):

| Variant | Cleaning | Stemmed | Use in |
|---|---|---|---|
| `*_clean` | HTML strip, whitespace norm, ASCII | No | FinBERT, BERTopic |
| `*_stemmed` | Clean + stopword removal + Snowball | Yes | LM dictionary, TF-IDF cosine sim |

**Also produces** `text_chunks.parquet`: clean text split into 400-token windows with 50-token overlap,
ready for batch FinBERT inference.

**Output files** (written to Drive):
- `text_preprocessed.parquet` — one row per (cik, year), both variants per section
- `text_chunks.parquet` — one row per (cik, year, section, chunk_idx)

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    # Folder containing {cik}_filings.parquet files (output of SEC_10K_Extractor_Drive)
    'filings_folder': '/content/drive/MyDrive/FML_project_4',

    # Where to write preprocessed outputs
    'output_folder': '/content/drive/MyDrive/FML_project_4',

    # Sections to process
    'sections': ['item_1', 'item_1A', 'item_7'],

    # FinBERT tokenizer used for accurate chunk token counting
    'tokenizer_name': 'ProsusAI/finbert',

    # BERT chunk size (tokens) and overlap
    'chunk_size': 400,
    'chunk_overlap': 50,

    # Hard cap on raw section length to avoid memory issues (chars)
    'max_section_chars': 150_000,

    'start_year': 2004,
    'end_year':   2025,
}
print('Config OK')

## 2. Install dependencies

In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', '-q', 'nltk', 'transformers', 'pyarrow', 'tqdm'],
    check=True,
)

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
print('Dependencies ready')

## 3. Load parquet files

In [ ]:
import glob
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import gc

SECTIONS: list[str] = CONFIG['sections']

pq_files = sorted(
    glob.glob(os.path.join(CONFIG['filings_folder'], '**', '*_filings.parquet'), recursive=True)
)
print(f'Parquet files found: {len(pq_files)}')

## 4. Cleaning pipeline

Strips HTML tags, normalises whitespace, removes non-ASCII characters, and
collapses repeated punctuation. **Does NOT stem** — the clean text is used
as-is by FinBERT and BERTopic.

In [ ]:
import re

# Pre-compile patterns for speed
_HTML_TAG    = re.compile(r'<[^>]+>')
_HTML_ENTITY = re.compile(r'&[a-zA-Z]{2,6};|&#\d{1,5};')
_NON_ASCII   = re.compile(r'[^\x00-\x7F]+')
_WHITESPACE  = re.compile(r'[ \t]+')
_MULTILINE   = re.compile(r'\n{3,}')
# SEC boilerplate patterns: page numbers, table of contents refs
_PAGE_NUM    = re.compile(r'\n\s*\d{1,3}\s*\n')
_EXHIBIT_REF = re.compile(r'(?i)exhibit\s+\d+\.\d+')


def clean_text(raw: str, max_chars: int) -> str:
    """Strip HTML, normalise whitespace and encoding. No stemming."""
    # Hard cap before any processing to avoid huge allocations
    text = raw[:max_chars]
    text = _HTML_TAG.sub(' ', text)
    text = _HTML_ENTITY.sub(' ', text)
    text = _NON_ASCII.sub(' ', text)
    text = _PAGE_NUM.sub('\n', text)
    text = _WHITESPACE.sub(' ', text)
    text = _MULTILINE.sub('\n\n', text)
    return text.strip()


# Quick smoke test
sample = '<p>This &amp; that\u2019s fine.\n\n\n\n   </p>  Page 42 '
assert 'This' in clean_text(sample, max_chars=1000)
print('clean_text OK')

## 5. Stemming pipeline

Applies Snowball stemmer after stopword removal. Used for Loughran-McDonald
scoring and TF-IDF cosine similarity (Track A and B in `nlp_features.ipynb`).

In [ ]:
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

_STOPWORDS: frozenset[str] = frozenset(stopwords.words('english'))
_STEMMER   = SnowballStemmer('english')
_WORD_RE   = re.compile(r'[a-zA-Z]+')


def stem_text(clean: str) -> str:
    """Tokenise, remove stopwords, apply Snowball stemmer. Returns space-joined tokens."""
    tokens = _WORD_RE.findall(clean.lower())
    stemmed = [
        _STEMMER.stem(tok)
        for tok in tokens
        if tok not in _STOPWORDS and len(tok) > 2
    ]
    return ' '.join(stemmed)


# Smoke test
assert 'compani' in stem_text('The companies are managing their risk factors.')
print('stem_text OK')

## 6. Streaming pipeline: clean → stem → chunk → save

Processes **80 parquet files per batch** (~800–1 000 filings). Each batch is fully
processed and written to disk before the next one is loaded — RAM never holds more
than one batch at a time.

Both output files (`text_preprocessed.parquet`, `text_chunks.parquet`) are built
incrementally via `pyarrow.ParquetWriter`, which avoids a final in-memory concat.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
from transformers import AutoTokenizer

# ── Tokenizer ──
tokenizer = AutoTokenizer.from_pretrained(CONFIG['tokenizer_name'])
CHUNK_SIZE: int = CONFIG['chunk_size']
STRIDE: int     = CONFIG['chunk_overlap']
DOC_BATCH: int  = 256   # docs per tokenizer batch (controls peak RAM during chunking)

# ── Streaming batch size ──
BATCH_FILES: int = 80   # parquet files per outer loop iteration
load_cols = ['cik', 'year', 'filing_date'] + SECTIONS

PRE_PATH    = os.path.join(CONFIG['output_folder'], 'text_preprocessed.parquet')
CHUNKS_PATH = os.path.join(CONFIG['output_folder'], 'text_chunks.parquet')

file_batches = [pq_files[i : i + BATCH_FILES] for i in range(0, len(pq_files), BATCH_FILES)]
print(f'{len(pq_files)} files → {len(file_batches)} batches of ≤{BATCH_FILES}')

pre_writer:   pq.ParquetWriter | None = None
chunk_writer: pq.ParquetWriter | None = None

try:
    for batch_num, batch_paths in enumerate(tqdm(file_batches, desc='Batches')):

        # ── 1. Load this batch ──────────────────────────────────────────────
        dfs: list[pd.DataFrame] = []
        for p in batch_paths:
            try:
                dfs.append(pd.read_parquet(p, columns=load_cols))
            except Exception as exc:
                print(f'  Skip {p}: {exc}')
        if not dfs:
            continue

        batch = pd.concat(dfs, ignore_index=True)
        del dfs

        batch['cik']  = batch['cik'].astype(str).str.strip().str.lstrip('0')
        batch['year'] = pd.to_numeric(batch['year'], errors='coerce').astype('Int64')
        batch = batch[batch['year'].between(CONFIG['start_year'], CONFIG['end_year'])]
        for sec in SECTIONS:
            if sec not in batch.columns:
                batch[sec] = ''
            batch[sec] = batch[sec].fillna('').astype(str)
        batch = (
            batch.sort_values('filing_date', na_position='first')
                 .drop_duplicates(subset=['cik', 'year'], keep='last')
                 .reset_index(drop=True)
        )

        # ── 2. Clean (vectorised pandas — runs in C, near-zero overhead) ────
        for sec in SECTIONS:
            batch[f'{sec}_clean'] = clean_series(batch[sec])

        # Drop raw text immediately — largest memory release in the pipeline
        batch.drop(columns=SECTIONS, inplace=True)
        gc.collect()

        # ── 3. Stem (ProcessPoolExecutor across all cores) ───────────────────
        N_WORKERS  = os.cpu_count() or 2
        STEM_BATCH = 200

        def _parallel_stem(texts: list[str]) -> list[str]:
            batches = [texts[i : i + STEM_BATCH] for i in range(0, len(texts), STEM_BATCH)]
            out: list[str] = [''] * len(texts)
            with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
                futs = {pool.submit(_stem_batch, b): i for i, b in enumerate(batches)}
                for fut in as_completed(futs):
                    i = futs[fut]
                    start = i * STEM_BATCH
                    for j, v in enumerate(fut.result()):
                        out[start + j] = v
            return out

        for sec in SECTIONS:
            batch[f'{sec}_stemmed'] = _parallel_stem(batch[f'{sec}_clean'].tolist())

        # ── 4. Write preprocessed rows ───────────────────────────────────────
        pre_cols = (
            ['cik', 'year', 'filing_date']
            + [f'{s}_clean'   for s in SECTIONS]
            + [f'{s}_stemmed' for s in SECTIONS]
        )
        pre_table = pa.Table.from_pandas(batch[pre_cols], preserve_index=False)
        if pre_writer is None:
            pre_writer = pq.ParquetWriter(PRE_PATH, pre_table.schema, compression='snappy')
        pre_writer.write_table(pre_table)
        del pre_table

        # ── 5. Tokenise + write chunks ───────────────────────────────────────
        for sec in SECTIONS:
            texts = batch[f'{sec}_clean'].tolist()
            ciks  = batch['cik'].tolist()
            years = batch['year'].tolist()

            chunk_rows: list[dict] = []
            for doc_start in range(0, len(texts), DOC_BATCH):
                enc = tokenizer(
                    texts[doc_start : doc_start + DOC_BATCH],
                    max_length=CHUNK_SIZE,
                    stride=STRIDE,
                    return_overflowing_tokens=True,
                    truncation=True,
                    padding=False,
                    add_special_tokens=False,
                )
                ctr: dict[int, int] = defaultdict(int)
                for ids, si in zip(enc['input_ids'], enc['overflow_to_sample_mapping']):
                    gi = doc_start + si   # global index within this file-batch
                    chunk_rows.append({
                        'cik':       ciks[gi],
                        'year':      years[gi],
                        'section':   sec,
                        'chunk_idx': ctr[gi],
                        'text':      tokenizer.decode(ids, skip_special_tokens=True),
                        'n_tokens':  len(ids),
                    })
                    ctr[gi] += 1

            if chunk_rows:
                chunk_df             = pd.DataFrame(chunk_rows)
                chunk_df['year']     = chunk_df['year'].astype('Int64')
                chunk_table          = pa.Table.from_pandas(chunk_df, preserve_index=False)
                if chunk_writer is None:
                    chunk_writer = pq.ParquetWriter(CHUNKS_PATH, chunk_table.schema, compression='snappy')
                chunk_writer.write_table(chunk_table)
                del chunk_df, chunk_table, chunk_rows

        del batch
        gc.collect()

finally:
    if pre_writer:   pre_writer.close()
    if chunk_writer: chunk_writer.close()

print('Pipeline complete.')
for path in (PRE_PATH, CHUNKS_PATH):
    size_mb = os.path.getsize(path) / 1e6
    meta    = pq.read_metadata(path)
    print(f'  {os.path.basename(path)}: {meta.num_rows:,} rows, {size_mb:.1f} MB')

## 7. Quality-control checks

Loads the saved files from disk (small column subsets only) to avoid pulling the full
text back into RAM.

In [ ]:
import matplotlib.pyplot as plt

# Load only word-count proxy columns — avoids pulling full text into RAM
wc_cols = ['cik', 'year'] + [f'{s}_clean' for s in SECTIONS]
pre_sample = pd.read_parquet(PRE_PATH, columns=wc_cols)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, sec in zip(axes, SECTIONS):
    col = f'{sec}_clean'
    word_counts = pre_sample[col].str.split().str.len().dropna()
    ax.hist(word_counts.clip(upper=15_000), bins=40, color='#1f77b4', edgecolor='white', alpha=0.85)
    ax.axvline(word_counts.median(), color='red', linestyle='--',
               label=f'Median {word_counts.median():.0f}')
    ax.set_title(f'{sec} — word count')
    ax.set_xlabel('Words (capped at 15k)')
    ax.set_ylabel('# filings')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Clean section lengths across all filings', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'section_length_dist.png'), dpi=150, bbox_inches='tight')
plt.show()
del pre_sample

# Empty-section audit and chunk stats
# Load only lightweight columns from each file

pre_wc   = pd.read_parquet(PRE_PATH,    columns=['cik', 'year'] + [f'{s}_clean' for s in SECTIONS])
chunk_meta = pd.read_parquet(CHUNKS_PATH, columns=['cik', 'year', 'section', 'chunk_idx', 'n_tokens'])

print('Empty / near-empty sections (< 50 words after cleaning):')
for sec in SECTIONS:
    col   = f'{sec}_clean'
    short = (pre_wc[col].str.split().str.len() < 50).sum()
    pct   = short / len(pre_wc) * 100
    print(f'  {col}: {short:>4} filings ({pct:.1f}%)')

print('\nChunk counts per section (max chunk_idx per doc):')
chunk_stats = (
    chunk_meta.groupby(['cik', 'year', 'section'])['chunk_idx']
              .max().reset_index()
              .groupby('section')['chunk_idx']
              .agg(['mean', 'std', 'max'])
)
print(chunk_stats.round(1).to_string())

print('\nToken distribution per chunk:')
print(chunk_meta['n_tokens'].describe().round(1).to_string())

del pre_wc, chunk_meta

print('\ntext_preprocessed.parquet → use in nlp_features.ipynb (LM dict, TF-IDF)')
print('text_chunks.parquet       → use in finbert_features.ipynb (FinBERT inference)')

In [ ]:
from collections import defaultdict

# Process each section in batches of DOC_BATCH documents.
# HuggingFace tokenizes the whole batch in Rust, then returns overflow chunks
# with overflow_to_sample_mapping telling us which source doc each chunk came from.
DOC_BATCH: int = 256

chunk_rows: list[dict] = []

for sec in SECTIONS:
    clean_col = f'{sec}_clean'
    texts  = preprocessed[clean_col].tolist()
    ciks   = preprocessed['cik'].tolist()
    years  = preprocessed['year'].tolist()

    for batch_start in tqdm(range(0, len(texts), DOC_BATCH), desc=f'Chunking {sec}'):
        batch_texts = texts[batch_start : batch_start + DOC_BATCH]
        batch_ciks  = ciks [batch_start : batch_start + DOC_BATCH]
        batch_years = years[batch_start : batch_start + DOC_BATCH]

        enc = tokenizer(
            batch_texts,
            max_length=CHUNK_SIZE,
            stride=STRIDE,
            return_overflowing_tokens=True,
            truncation=True,
            padding=False,
            add_special_tokens=False,
        )

        # overflow_to_sample_mapping[i] = index within this batch of the source doc
        chunk_counter: dict[int, int] = defaultdict(int)
        for ids, sample_idx in zip(enc['input_ids'], enc['overflow_to_sample_mapping']):
            chunk_rows.append({
                'cik':       batch_ciks[sample_idx],
                'year':      batch_years[sample_idx],
                'section':   sec,
                'chunk_idx': chunk_counter[sample_idx],
                'text':      tokenizer.decode(ids, skip_special_tokens=True),
                'n_tokens':  len(ids),
            })
            chunk_counter[sample_idx] += 1

chunks_df = pd.DataFrame(chunk_rows)
chunks_df['year'] = chunks_df['year'].astype('Int64')

chunks_path = os.path.join(CONFIG['output_folder'], 'text_chunks.parquet')
chunks_df.to_parquet(chunks_path, index=False, compression='snappy')

size_mb = os.path.getsize(chunks_path) / 1e6
print(f'Saved text_chunks.parquet  {size_mb:.1f} MB')
print(f'Total chunks: {len(chunks_df):,}')
print(chunks_df.groupby('section')['chunk_idx'].max().rename('max_chunk_idx_per_doc').to_string())
chunks_df.head(4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, sec in zip(axes, SECTIONS):
    col = f'{sec}_clean'
    word_counts = preprocessed[col].str.split().str.len().dropna()
    ax.hist(word_counts.clip(upper=15_000), bins=40, color='#1f77b4', edgecolor='white', alpha=0.85)
    ax.axvline(word_counts.median(), color='red', linestyle='--',
               label=f'Median {word_counts.median():.0f}')
    ax.set_title(f'{sec} — word count distribution')
    ax.set_xlabel('Words (capped at 15k)')
    ax.set_ylabel('# filings')
    ax.legend(fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Clean section lengths across all filings', fontsize=13)
plt.tight_layout()
plt.savefig(
    os.path.join(CONFIG['output_folder'], 'section_length_dist.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()